# 03 - Modelagem, IA e Sistemas Inteligentes

Notebook responsável pelas análises avançadas, modelagem estatística, inteligência artificial, sistemas de recomendação e experimentos de IA generativa aplicados aos dados de compras.

Objetivos:
- Market Basket Analysis
- Sistemas de Recomendação
- Embeddings de Produtos
- IA Generativa
- Insights Automáticos
- Modelagem Temporal

In [34]:
# !pip install -r requirements.txt

In [35]:
# IMPORTS
import time
from pathlib import Path
import pandas as pd
import warnings

from src.core.tictoc import tictoc
from src.core.config_loader import ConfigLoader
from src.core.paths import ROOT_DIR
from src.core.logger import setup_logger
from src.core.model import *

# CONFIGURAÇÃO
warnings.filterwarnings('ignore')
logger = setup_logger(__name__)
tic_geral = time.time()

config = ConfigLoader(
    "config/settings.yaml"
)

## 1. Carregamento dos Dados

Nesta etapa é realizado o carregamento da base analítica consolidada utilizada nos experimentos de modelagem estatística, inteligência artificial, sistemas de recomendação e análises avançadas de comportamento de consumo.

| Variável             | Tipo                | Descrição                             |
| -------------------- | ------------------- | ------------------------------------- |
| chave_anonimizada    | categórica          | Identificador único da nota fiscal    |
| data_hora            | datetime            | Data e hora da compra                 |
| mes_ano              | categórica temporal | Ano e mês da compra                   |
| dia_semana           | categórica          | Dia da semana da compra               |
| feriado              | binária             | Indica se a compra ocorreu em feriado |
| estacao_ano          | categórica          | Estação climática                     |
| temperatura_max      | numérica contínua   | Temperatura máxima do dia             |
| temperatura_min      | numérica contínua   | Temperatura mínima do dia             |
| temperatura_media    | numérica contínua   | Temperatura média do dia              |
| chuva_mm             | numérica contínua   | Volume de chuva                       |
| cat_temperatura      | categórica ordinal  | Faixa de temperatura                  |
| dia_chuvoso          | binária             | Indica ocorrência de chuva            |
| periodo_dia          | categórica ordinal  | Madrugada, manhã, tarde ou noite      |
| cnpj                 | categórica          | Identificador do supermercado         |
| supermercado         | categórica          | Nome do supermercado                  |
| qtd_total_nota       | numérica contínua   | Quantidade total de produtos da nota  |
| valor_total_nota     | numérica contínua   | Valor total da nota fiscal            |
| valor_total_tributos | numérica contínua   | Soma dos tributos da nota             |
| cod_produto          | categórica          | Código do produto                     |
| categoria_produto    | categórica          | Categoria do produto                  |
| produto              | textual/categórica  | Nome do produto                       |
| unidade              | categórica          | Unidade de venda                      |
| quantidade           | numérica contínua   | Quantidade comprada                   |
| preco_unitario       | numérica contínua   | Preço unitário                        |
| preco_total          | numérica contínua   | Valor total do item                   |

In [36]:
tic = time.time()

In [37]:
data_path = config.get(
    "paths.data"
)
df = pd.read_excel(data_path)
print(
    f"Linhas: {df.shape[0]}"
)
print(
    f"Colunas: {df.shape[1]}"
)
df.head()

Linhas: 4074
Colunas: 25


,chave_anonimizada,data_hora,mes_ano,dia_semana,feriado,estacao_ano,temperatura_max,temperatura_min,temperatura_media,chuva_mm,...,qtd_total_nota,valor_total_nota,valor_total_tributos,cod_produto,categoria_produto,produto,unidade,quantidade,preco_unitario,preco_total
0,fcb669525bd643f21cb51d896c86adfe0933f5cb868f32...,2022-06-28 20:44:50,2022-06,TERCA,NAO,INVERNO,21.3,9.8,14.7,0.0,...,82,679.79,76.32,10166630001,MASSA,NISSIN TALH 99G BOL,UNIDADE,6.0,2.69,16.14
1,fcb669525bd643f21cb51d896c86adfe0933f5cb868f32...,2022-06-28 20:44:50,2022-06,TERCA,NAO,INVERNO,21.3,9.8,14.7,0.0,...,82,679.79,76.32,11115310001,UTILIDADES,VELA PARA N3 C/6 BCA,UNIDADE,1.0,5.98,5.98
2,fcb669525bd643f21cb51d896c86adfe0933f5cb868f32...,2022-06-28 20:44:50,2022-06,TERCA,NAO,INVERNO,21.3,9.8,14.7,0.0,...,82,679.79,76.32,11158780001,BEBIDA NAO ALCOOLICA,BEB MONSTER 473ML MG,UNIDADE,1.0,8.49,8.49
3,fcb669525bd643f21cb51d896c86adfe0933f5cb868f32...,2022-06-28 20:44:50,2022-06,TERCA,NAO,INVERNO,21.3,9.8,14.7,0.0,...,82,679.79,76.32,11158780001,BEBIDA NAO ALCOOLICA,BEB MONSTER 473ML MG,UNIDADE,1.0,8.49,8.49
4,fcb669525bd643f21cb51d896c86adfe0933f5cb868f32...,2022-06-28 20:44:50,2022-06,TERCA,NAO,INVERNO,21.3,9.8,14.7,0.0,...,82,679.79,76.32,248736,BOMBONIERE,BOMBOM LACTA 250 6G,UNIDADE,1.0,8.99,8.99


In [38]:
toc = time.time()
print(f"\n\033[33mEtapa 1 | Tempo:\033[0;0m {tictoc(tic, toc)}")


Etapa 1 | Tempo: (00:00:04) [04/11/2026 00:27:51 - 04/11/2026 00:27:55]


### 2. Entendimento Inicial dos Dados

**POSSÍVEIS VARIÁVEIS RESPOSTA - MODELOS SUPERVISIONADOS**

**Regressão**
| variável resposta | objetivo              |
| ----------------- | --------------------- |
| preco_unitario    | previsão de preço     |
| preco_total       | previsão de gasto     |
| valor_total_nota  | previsão de ticket    |
| quantidade        | previsão de consumo   |
| chuva_mm          | inferência contextual |

**Classificação**
| variável resposta | objetivo                                            |
| ----------------- | --------------------------------------------------- |
| feriado           | prever padrão feriado                               |
| categoria_produto | classificação textual                               |
| produto           | prever a compra de determinado produto (ex: Whisky) |

---

**MODELOS NÃO SUPERVISIONADOS**

**Clusterização**
| técnica | objetivo            |
| ------- | ------------------- |
| KMeans  | tipos de compra     |
| HDBSCAN | padrões raros       |
| PCA     | redução dimensional |
| UMAP    | embeddings visuais  |

---

**ASSOCIATION RULES**

| técnica   | objetivo                  |
| --------- | ------------------------- |
| FP-Growth | produtos comprados juntos |
| Apriori   | cesta                     |
| Lift      | relações fortes           |

---

**NLP / IA**

| técnica               | objetivo             |
| --------------------- | -------------------- |
| Embeddings            | similaridade         |
| LLM                   | insights automáticos |
| RAG                   | consulta inteligente |
| Recommendation System | recomendação         |

In [39]:
tic = time.time()

In [40]:
results = analyze_dataset(df)

2026-11-04 00:27:55,374 | src.core.model | INFO | Iniciando análise exploratória da base
2026-11-04 00:27:55,588 | src.core.model | INFO | Análise exploratória concluída


#### 2.1 Shape

In [41]:
results['shape']

{'linhas': 4074, 'colunas': 25}

#### 2.2 Nulos

In [42]:
results['missing']

,total_nulos,percentual_nulos
valor_total_tributos,22,0.54
data_hora,0,0.00
mes_ano,0,0.00
dia_semana,0,0.00
feriado,0,0.00
estacao_ano,0,0.00
temperatura_max,0,0.00
temperatura_min,0,0.00
chave_anonimizada,0,0.00
temperatura_media,0,0.00


#### 2.3 Estatísticas Numéricas

In [43]:
results['numeric_description']

,count,mean,std,min,25%,50%,75%,max
temperatura_max,4074.0,2.241848e+01,3.902910e+00,11.700,20.60,22.500,25.30,3.240000e+01
temperatura_min,4074.0,1.389212e+01,3.443954e+00,2.200,10.80,13.900,16.80,2.070000e+01
temperatura_media,4074.0,1.745565e+01,3.380476e+00,7.300,15.30,17.200,20.30,2.620000e+01
chuva_mm,4074.0,3.653436e+00,6.698458e+00,0.000,0.00,1.000,3.50,6.310000e+01
qtd_total_nota,4074.0,4.488807e+01,3.368144e+01,1.000,18.00,33.000,71.00,1.180000e+02
valor_total_nota,4074.0,3.836637e+02,2.962403e+02,3.490,140.58,268.920,648.32,9.995600e+02
valor_total_tributos,4052.0,1.028107e+02,9.469776e+01,0.000,18.15,76.320,169.39,3.234700e+02
cod_produto,4074.0,1.155981e+09,3.332394e+09,74.000,99462.00,267260.000,1199039.00,1.204739e+10
quantidade,4074.0,1.264189e+00,1.443793e+00,0.035,1.00,1.000,1.00,1.400000e+01
preco_unitario,4074.0,8.279107e+00,8.904776e+00,0.690,3.39,5.990,9.19,1.499000e+02


#### 2.4 Cardinalidade

In [44]:
results['cardinality']

,variavel,qtd_unicos
20,produto,1281
18,cod_produto,1281
24,preco_total,774
23,preco_unitario,502
0,chave_anonimizada,269
1,data_hora,269
16,valor_total_nota,244
22,quantidade,238
17,valor_total_tributos,228
6,temperatura_max,120


#### 2.5 Correlação

In [45]:
results['correlation']

,temperatura_max,temperatura_min,temperatura_media,chuva_mm,dia_chuvoso,qtd_total_nota,valor_total_nota,valor_total_tributos,cod_produto,quantidade,preco_unitario,preco_total
temperatura_max,1.000000,0.748871,0.919557,-0.028026,-0.030405,-0.190640,-0.079344,-0.042417,0.069747,0.045552,0.007673,0.039716
temperatura_min,0.748871,1.000000,0.929509,0.264223,0.401876,-0.219605,-0.118059,-0.062170,0.002042,0.049873,-0.000206,0.032290
temperatura_media,0.919557,0.929509,1.000000,0.146845,0.219934,-0.244693,-0.120332,-0.072745,0.045584,0.050651,0.007018,0.042884
chuva_mm,-0.028026,0.264223,0.146845,1.000000,0.320603,-0.118704,-0.061535,-0.031769,-0.043094,0.029069,0.015481,0.033112
dia_chuvoso,-0.030405,0.401876,0.219934,0.320603,1.000000,-0.084710,-0.034711,0.001264,-0.068455,0.037132,-0.022255,0.000150
qtd_total_nota,-0.190640,-0.219605,-0.244693,-0.118704,-0.084710,1.000000,0.934532,0.894459,-0.166274,-0.019045,-0.033576,-0.034656
valor_total_nota,-0.079344,-0.118059,-0.120332,-0.061535,-0.034711,0.934532,1.000000,0.954490,-0.188545,0.021616,0.019565,0.042200
valor_total_tributos,-0.042417,-0.062170,-0.072745,-0.031769,0.001264,0.894459,0.954490,1.000000,-0.291368,0.010514,0.023616,0.043333
cod_produto,0.069747,0.002042,0.045584,-0.043094,-0.068455,-0.166274,-0.188545,-0.291368,1.000000,0.055933,-0.035832,-0.015513
quantidade,0.045552,0.049873,0.050651,0.029069,0.037132,-0.019045,0.021616,0.010514,0.055933,1.000000,-0.126607,0.244997


#### 2.7 Outliers

In [46]:
pd.DataFrame(results['outliers']).T

,total_outliers,percentual_outliers
temperatura_max,55.0,1.35
temperatura_min,0.0,0.00
temperatura_media,6.0,0.15
chuva_mm,492.0,12.08
qtd_total_nota,0.0,0.00
valor_total_nota,0.0,0.00
valor_total_tributos,0.0,0.00
cod_produto,860.0,21.11
quantidade,669.0,16.42
preco_unitario,384.0,9.43


#### 2.8 Categorias mais frequentes

In [47]:
results['categorical_summary']['categoria_produto']

categoria_produto
BEBIDA NAO ALCOOLICA    1415
MERCEARIA                467
PADARIA E FRIOS          303
CONGELADO                239
BOMBONIERE               229
MASSA                    225
HIGIENE E LIMPEZA        201
BEBIDA ALCOOLICA         170
MOLHO                    167
HORTIFRUTI               122
Name: count, dtype: int64

#### 2.9 Intervalo Temporal

In [48]:
results['date_range']

{'data_min': Timestamp('2022-06-28 20:44:50'),
 'data_max': Timestamp('2026-11-02 15:36:30'),
 'dias': 1587}

In [49]:
toc = time.time()
print(f"\n\033[33mEtapa 2 | Tempo:\033[0;0m {tictoc(tic, toc)}")


Etapa 2 | Tempo: (00:00:00) [04/11/2026 00:27:55 - 04/11/2026 00:27:56]


### 3. Feature Engineering

In [50]:
tic = time.time()

In [51]:
# ==========================================================
# FEATURE ENGINEERING
# ==========================================================

df_model = feature_engineering(df)

print(
    f'Colunas originais: {df.shape[1]}'
)

print(
    f'Colunas após FE: {df_model.shape[1]}'
)

2026-11-04 00:27:56,232 | src.core.model | INFO | Iniciando feature engineering
2026-11-04 00:27:56,255 | src.core.model | INFO | Criando features temporais
2026-11-04 00:27:56,274 | src.core.model | INFO | Criando features financeiras
2026-11-04 00:27:56,283 | src.core.model | INFO | Criando features de frequência
2026-11-04 00:27:56,306 | src.core.model | INFO | Criando features de preço
2026-11-04 00:27:56,314 | src.core.model | INFO | Criando features por nota fiscal
2026-11-04 00:27:56,326 | src.core.model | INFO | Criando features climáticas
2026-11-04 00:27:56,333 | src.core.model | INFO | Criando variáveis normalizadas
2026-11-04 00:27:56,358 | src.core.model | INFO | Criando features textuais
2026-11-04 00:27:56,375 | src.core.model | INFO | Criando flags analíticas
2026-11-04 00:27:56,391 | src.core.model | INFO | Feature engineering concluído | Novas colunas: 31


Colunas originais: 25
Colunas após FE: 56


#### 3.1 Novas Variáveis

In [52]:
df_model[[
    'produto',
    'produto_clean',
    'ticket_medio_item',
    'percentual_tributo',
    'produto_promocao',
    'qtd_itens_nota'
]].head()

,produto,produto_clean,ticket_medio_item,percentual_tributo,produto_promocao,qtd_itens_nota
0,NISSIN TALH 99G BOL,NISSIN TALH 99G BOL,8.290122,0.11227,0,82
1,VELA PARA N3 C/6 BCA,VELA PARA N3 C6 BCA,8.290122,0.11227,0,82
2,BEB MONSTER 473ML MG,BEB MONSTER 473ML MG,8.290122,0.11227,0,82
3,BEB MONSTER 473ML MG,BEB MONSTER 473ML MG,8.290122,0.11227,0,82
4,BOMBOM LACTA 250 6G,BOMBOM LACTA 250 6G,8.290122,0.11227,0,82


#### 3.2 Sample

In [53]:
novas_colunas = [

    col for col in df_model.columns

    if col not in df.columns
]

novas_colunas

['ano',
 'mes',
 'dia',
 'hora',
 'dia_ano',
 'semana_ano',
 'fim_semana',
 'ticket_medio_item',
 'percentual_tributo',
 'preco_por_quantidade',
 'frequencia_produto',
 'frequencia_categoria',
 'frequencia_supermercado',
 'desvio_preco_produto',
 'percentual_desvio_preco',
 'produto_promocao',
 'qtd_itens_nota',
 'qtd_categorias_nota',
 'amplitude_termica',
 'chuva_intensa',
 'preco_unitario_scaled',
 'preco_total_scaled',
 'quantidade_scaled',
 'valor_total_nota_scaled',
 'temperatura_media_scaled',
 'chuva_mm_scaled',
 'produto_clean',
 'produto_tamanho_texto',
 'compra_grande',
 'produto_caro',
 'produto_barato']

#### 3.3 Promoções Detectadas

In [54]:
df_model[
    df_model['produto_promocao'] == 1
][[
    'produto',
    'preco_unitario',
    'percentual_desvio_preco'
]].head(20)

,produto,preco_unitario,percentual_desvio_preco
47,POPCOR YOKI 100G BAC,3.98,-0.168234
66,P AL WYDA 30X4MT,2.99,-0.167131
69,FANDANGOS 45G QJO,2.69,-0.154088
70,FANDANGOS 45G QJO,2.69,-0.154088
89,BEB NESQUIK 200ML MO,2.29,-0.276207
119,TOMATE ITALIANO kg,2.99,-0.505422
131,LIMAO THAITI kg,1.99,-0.412979
134,FANDANGOS 45G QJO,2.69,-0.154088
141,MORANGO FRES 250G,7.89,-0.160192
145,AMIDO M MAIZENA 200G,3.99,-0.210943


#### 3.4 Correlação Novas

In [55]:
df_model.corr(
    numeric_only=True
)

,temperatura_max,temperatura_min,temperatura_media,chuva_mm,dia_chuvoso,qtd_total_nota,valor_total_nota,valor_total_tributos,cod_produto,quantidade,...,preco_unitario_scaled,preco_total_scaled,quantidade_scaled,valor_total_nota_scaled,temperatura_media_scaled,chuva_mm_scaled,produto_tamanho_texto,compra_grande,produto_caro,produto_barato
temperatura_max,1.000000,0.748871,0.919557,-0.028026,-0.030405,-0.190640,-0.079344,-0.042417,6.974705e-02,0.045552,...,0.007673,0.039716,0.045552,-0.079344,0.919557,-0.028026,-5.063131e-02,-0.053879,-0.012832,0.012583
temperatura_min,0.748871,1.000000,0.929509,0.264223,0.401876,-0.219605,-0.118059,-0.062170,2.042282e-03,0.049873,...,-0.000206,0.032290,0.049873,-0.118059,0.929509,0.264223,-6.450835e-02,-0.111659,-0.000230,0.029084
temperatura_media,0.919557,0.929509,1.000000,0.146845,0.219934,-0.244693,-0.120332,-0.072745,4.558384e-02,0.050651,...,0.007018,0.042884,0.050651,-0.120332,1.000000,0.146845,-5.581202e-02,-0.123458,-0.002973,0.023101
chuva_mm,-0.028026,0.264223,0.146845,1.000000,0.320603,-0.118704,-0.061535,-0.031769,-4.309444e-02,0.029069,...,0.015481,0.033112,0.029069,-0.061535,0.146845,1.000000,-5.739848e-03,0.059866,0.024021,0.010765
dia_chuvoso,-0.030405,0.401876,0.219934,0.320603,1.000000,-0.084710,-0.034711,0.001264,-6.845466e-02,0.037132,...,-0.022255,0.000150,0.037132,-0.034711,0.219934,0.320603,-5.614951e-02,-0.186393,0.006103,0.043519
qtd_total_nota,-0.190640,-0.219605,-0.244693,-0.118704,-0.084710,1.000000,0.934532,0.894459,-1.662744e-01,-0.019045,...,-0.033576,-0.034656,-0.019045,0.934532,-0.244693,-0.118704,-2.399439e-01,0.563939,-0.013981,-0.015330
valor_total_nota,-0.079344,-0.118059,-0.120332,-0.061535,-0.034711,0.934532,1.000000,0.954490,-1.885452e-01,0.021616,...,0.019565,0.042200,0.021616,1.000000,-0.120332,-0.061535,-2.343933e-01,0.649732,0.032445,-0.031840
valor_total_tributos,-0.042417,-0.062170,-0.072745,-0.031769,0.001264,0.894459,0.954490,1.000000,-2.913680e-01,0.010514,...,0.023616,0.043333,0.010514,0.954490,-0.072745,-0.031769,-2.513805e-01,0.653163,0.025415,-0.010397
cod_produto,0.069747,0.002042,0.045584,-0.043094,-0.068455,-0.166274,-0.188545,-0.291368,1.000000e+00,0.055933,...,-0.035832,-0.015513,0.055933,-0.188545,0.045584,-0.043094,2.948177e-02,-0.125417,-0.034058,0.019502
quantidade,0.045552,0.049873,0.050651,0.029069,0.037132,-0.019045,0.021616,0.010514,5.593339e-02,1.000000,...,-0.126607,0.244997,1.000000,0.021616,0.050651,0.029069,7.016582e-02,0.006077,-0.080282,0.263248


In [56]:
toc = time.time()
print(f"\n\033[33mEtapa 3 | Tempo:\033[0;0m {tictoc(tic, toc)}")


Etapa 3 | Tempo: (00:00:00) [04/11/2026 00:27:56 - 04/11/2026 00:27:56]


### 4. Market Basket Analysis

Nesta etapa são identificadas relações entre produtos e categorias frequentemente comprados em conjunto utilizando técnicas de associação.

In [57]:
tic = time.time()

In [58]:
# ==========================================================
# MARKET BASKET ANALYSIS
# ==========================================================

basket_results = run_market_basket_analysis(

    df=df_model,

    min_support=0.01,

    metric='lift',

    min_threshold=1.2,

    max_len=3,

    top_n_rules=20
)

2026-11-04 00:27:56,981 | src.core.model | INFO | Iniciando Market Basket Analysis
2026-11-04 00:27:56,992 | src.core.model | INFO | Realizando limpeza dos dados
2026-11-04 00:27:56,998 | src.core.model | INFO | Selecionando produtos mais relevantes
2026-11-04 00:27:57,012 | src.core.model | INFO | Construindo basket matrix
2026-11-04 00:27:57,026 | src.core.model | INFO | Executando FP-Growth
2026-11-04 00:27:57,650 | src.core.model | INFO | Gerando regras de associação
2026-11-04 00:27:58,999 | src.core.model | INFO | Calculando frequência dos produtos
2026-11-04 00:27:59,007 | src.core.model | INFO | Calculando métricas resumidas
2026-11-04 00:27:59,011 | src.core.model | INFO | Market Basket Analysis concluído


#### 4.1 Resumo Geral

In [59]:
basket_results['metrics_summary']

{'total_notas': 216,
 'total_produtos': 150,
 'total_regras': 73128,
 'media_lift': np.float64(9.4739),
 'media_confidence': np.float64(0.4498),
 'media_support': np.float64(0.0171)}

#### 4.2 Regras Mais Fortes

In [60]:
basket_results['top_rules'][[
    'antecedents',
    'consequents',
    'support',
    'confidence',
    'lift'
]]

,antecedents,consequents,support,confidence,lift
0,"AGUA COCO SOCO 200ML, BEB GATORADE 500ML",AG COC SOCOCO 200ML,0.013889,1.0,43.2
1,"CEBOLA KG, BEB LAC TODD 200ML",ENERG RED BULL 250ML,0.013889,1.0,43.2
2,"MILHO QUERO 170G, BEB LAC DANONE 250ML",ENERG RED BULL 250ML,0.013889,1.0,43.2
3,"BEB LAC DANONE 250ML, REFRIG COCA COLA 2L",ENERG RED BULL 250ML,0.013889,1.0,43.2
4,"SALG ELMA CHIPS 35G, REFRIG COCA COLA 2L",ENERG RED BULL 250ML,0.013889,1.0,43.2
5,"BEB LAC TODD 200ML, BEB LAC DANONE 250ML",ENERG RED BULL 250ML,0.013889,1.0,43.2
6,"BEB LAC TODD 200ML, SALG ELMA CHIPS 35G",ENERG RED BULL 250ML,0.013889,1.0,43.2
7,"LEITE COND PIRA 395G, SALG ELMA CHIPS 35G",ENERG RED BULL 250ML,0.013889,1.0,43.2
8,"LEITE COND PIRA 395G, BEB LAC TODD 200ML",ENERG RED BULL 250ML,0.013889,1.0,43.2
9,AG COC SOCOCO 200ML,"AGUA COCO SOCO 200ML, BEB GATORADE 500ML",0.013889,0.6,43.2


#### 4.3 Itemsets Frequentes

In [61]:
basket_results['frequent_items']

,support,itemsets
0,0.138889,frozenset({ENERG MONST 473ML})
1,0.134259,frozenset({CEBOLA KG})
2,0.129630,frozenset({LEITE TIROL TP 1L})
3,0.120370,frozenset({DROPS HALLS 28G})
4,0.115741,frozenset({LIMAO KG})
...,...,...
13811,0.013889,"frozenset({BEB LAC YOPRO 250ML, AGUA CRIST AZ ..."
13812,0.013889,"frozenset({SUCO DEL VALLE 1L, AGUA CRIST AZ 51..."
13813,0.013889,"frozenset({BEB LAC YOPRO 250ML, BEB GATORADE 5..."
13814,0.013889,"frozenset({SUCO DEL VALLE 1L, BEB GATORADE 500..."


#### 4.4 Produtos Mais Frequentes

In [62]:
basket_results['product_frequency'].head(20)

,produto,frequencia
0,BEB LAC YOPRO 250ML,57
1,ENERG MONST 473ML,47
2,BEB LAC DANONE 250ML,47
3,SUCO MAIS MARACUJ 1L,46
4,DROPS HALLS 28G,43
5,AGUA COCO SOCO 200ML,42
6,LEITE TIROL TP 1L,41
7,MAC NISSIN 99G,41
8,CREME LTE LIDER 200G,40
9,BEB NESQUIK 200ML MO,36


#### 4.5 Regras Muito Fortes

In [63]:
basket_results['rules'][
    basket_results['rules']['lift'] >= 3
][[
    'antecedents',
    'consequents',
    'lift',
    'confidence'
]]

,antecedents,consequents,lift,confidence
0,"AGUA COCO SOCO 200ML, BEB GATORADE 500ML",AG COC SOCOCO 200ML,43.2,1.000
1,"CEBOLA KG, BEB LAC TODD 200ML",ENERG RED BULL 250ML,43.2,1.000
2,"MILHO QUERO 170G, BEB LAC DANONE 250ML",ENERG RED BULL 250ML,43.2,1.000
3,"BEB LAC DANONE 250ML, REFRIG COCA COLA 2L",ENERG RED BULL 250ML,43.2,1.000
4,"SALG ELMA CHIPS 35G, REFRIG COCA COLA 2L",ENERG RED BULL 250ML,43.2,1.000
...,...,...,...,...
72447,WHISKY PASSP 670ML,"OVO MANTIQUEIRA UN, CREME LTE LIDER 200G",3.0,0.125
72448,WHISKY PASSP 670ML,"OVO MANTIQUEIRA UN, TOMATE KG",3.0,0.125
72449,WHISKY PASSP 670ML,"BATATA MONALISA KG, SUCO DEL VALLE 1L",3.0,0.125
72450,WHISKY PASSP 670ML,"BEB LAC YOPRO 250ML, CREME LTE LIDER 200G",3.0,0.125


In [64]:
toc = time.time()
print(f"\n\033[33mEtapa 4 | Tempo:\033[0;0m {tictoc(tic, toc)}")


Etapa 4 | Tempo: (00:00:02) [04/11/2026 00:27:56 - 04/11/2026 00:27:59]


### 5. NLP & Embeddings

Nesta etapa são gerados embeddings semânticos dos produtos para análises de similaridade, agrupamento e recomendação.

In [65]:
tic = time.time()

In [66]:
df['produto_clean']

KeyError: 'produto_clean'

In [ ]:
!pip install sentence_transformers

In [ ]:
from sentence_transformers import (
    SentenceTransformer
)

In [ ]:
model = SentenceTransformer(
    'all-MiniLM-L6-v2'
)

In [ ]:
embeddings = model.encode(
    df['produto'].unique()
)

In [ ]:
cosine_similarity(...)

In [ ]:
toc = time.time()
print(f"\n\033[33mEtapa 5 | Tempo:\033[0;0m {tictoc(tic, toc)}")

### 6. IA Generativa e Insights Automáticos

Nesta etapa são exploradas abordagens de LLMs para geração automática de insights, explicações estatísticas e interpretação dos padrões de consumo.

In [ ]:
tic = time.time()

In [ ]:
def generate_insight(...):

In [ ]:
toc = time.time()
print(f"\n\033[33mEtapa 6 | Tempo:\033[0;0m {tictoc(tic, toc)}")

### 8. Próximos Passos

- Sistemas de recomendação
- Forecast temporal
- Detecção de anomalias
- RAG sobre dados financeiros
- IA contextual com clima e inflação

## 3. Tempo Decorrido

In [ ]:
toc_geral = time.time()
print(f"\n\033[33mTempo decorrido no notebook:\033[0;0m {tictoc(tic_geral, toc_geral)}")